# Faza I – Para-procesimi i të dhënave

**Prokurimi publik në Shqipëri, 2016–2025** · Grupi 13 · Përgatitja dhe vizualizimi i të dhënave

Ky notebook tregon hap pas hapi punën e Fazës I, sipas tri grupeve të kërkesave të profesorit:

- **a)** Mbledhja e të dhënave, definimi i tipeve, kualiteti i të dhënave
- **b)** Integrimi, agregimi, mostrimi, pastrimi, identifikimi dhe strategjia e trajtimit për vlerat e zbrazëta
- **c)** Reduktimi i dimensionit, zgjedhja e nën-bashkësisë së vetive, krijimi i vetive, diskretizimi dhe binarizimi, transformimi

Notebook-u **nuk e rishkruan logjikën**: thërret funksionet e kodit në `src/`, të ndara në tri pjesë:

| Pjesa | Anëtari | Kodi | Dokumentimi |
|---|---|---|---|
| Vlerat dhe fituesit | Marigona Hamiti | `src/values_winners/` | `docs/values_winners.md` |
| Koha dhe integrimi | Semine Muharremi | `src/time_integration/` | `docs/time_integration.md` |
| Procedura dhe konkurrenca | Tringa Baftiu | `src/procedure_competition/` | `docs/procedure_competition.md` |

**Para ekzekutimit:** 10 skedarët CSV të APP-së (2016–2025) duhet të jenë te `data/raw/` (shih README). Ekzekutimi i plotë zgjat rreth 2 minuta.

In [41]:
import os
import sys
from pathlib import Path

# Notebook-u ekzekutohet nga rrënja e repository-t, që të gjenden src/ dhe data/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

DATA = Path("data/raw")

---
# a) Mbledhja, tipet dhe kualiteti

## 1. Mbledhja e të dhënave

Burimi: **Agjencia e Prokurimit Publik (APP)**, [Eksportimi i procedurave të publikuara](https://www.app.gov.al/eksportimi-i-procedurave-te-publikuara/). Një skedar CSV për çdo vit, 21 kolona. Viti 2026 përjashtohet, sepse në datën e eksportit nuk ishte i plotë. Të gjitha kolonat lexohen si tekst, që të ruhet formati origjinal.

**Objekti** i të dhënave është një procedurë prokurimi (tender): çdo rresht është një procedurë. Dataset-i është i tipit **record data**, me komponent kohor (datat) dhe kategori (autoriteti, procedura).

In [42]:
from src.common.data_loading import read_years

raw = read_years(DATA)
print(f"{len(raw):,} procedura x {raw.shape[1] - 1} kolona (+ kolona 'year')")
raw.groupby("year").size().rename("procedura").to_frame().T

277,764 procedura x 21 kolona (+ kolona 'year')


year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
procedura,33375,31769,33880,28313,23670,22653,26034,25137,27466,25467


In [43]:
# Rreshtat janë observimet (procedurat), kolonat janë variablat
raw.head(3)

,Autoriteti_kontraktues,Numri_i_references,Objekti_i_prokurimit,Lloji_i_procedures,Tipi_i_kontrates,Lloji_i_marreveshjes_kuader,Fondi_limit,Data_e_publikimit,Data_e_hapjes,Data_e_mbylljes,Anulluar,Arsyeja_e_anullimit,Pezulluar,Fituesi,NIPT_i_fituesit,Vlera_e_fituesit,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Lidhja_e_kontrates_me_TVSH,Numri_i_ofertave_te_dorezuara,Numri_i_ofertave_te_kualifikuara,Kodet_CPV,year
0,Bashkia Fushe Arrez,REF-56586-11-04-2016,Kolaudim punimesh ne objektin rikonstruksion rruga gjimnaz Fushe-Arrez,Small Value,Services,NaN,"28,875.00",04.11.2016,04.11.2016,07.11.2016,Jo,NaN,Jo,NOVATECH STUDIO,L01717030C,"19,900.00",NaN,Jo,6,6,NaN,2016
1,Sh.A Ujesjelles Kanalizime Fier,REF-34728-04-21-2016,Blerje materiale te ndryshme 2,Small Value,Supplies,NaN,"795,000.00",21.04.2016,21.04.2016,22.04.2016,Jo,NaN,Jo,MOENA AL 66,L43004402M,"529,900.00",NaN,Jo,22,22,NaN,2016
2,Drejtoria e Administrates dhe Sherbimeve ne M.M.,REF-37738-05-13-2016,1 Bilete avioni ne itinerarin TR ROME TR,Small Value,Services,NaN,"35,000.00",13.05.2016,13.05.2016,16.05.2016,Jo,NaN,Jo,ITRAVEL-ALBANIA EXPRESS,J61821074U,"26,960.00",NaN,Jo,4,4,NaN,2016


In [44]:
# Tipi që Pandas i ka dhënë çdo kolone: të gjitha janë tekst, sepse u lexuan si tekst
raw.dtypes.value_counts()

str      21
int64     1
Name: count, dtype: int64

### Kontrolli: a përputhet tipi i kolonës me tipin e vlerave?

Për çdo kolonë tekstuale kontrollojmë sa përqind e vlerave mund të kthehen në **numër** (pasi hiqet presja e mijësheve, `"741,600.00"` → `741600.00`) ose në **datë** (`DD.MM.YYYY`). Nëse pothuajse të gjitha vlerat kthehen, kolona në të vërtetë është numerike ose datë dhe duhet konvertuar te pastrimi.

In [45]:
text_cols = [c for c in raw.columns if c != "year"]
scan = []
for col in text_cols:
    values = raw[col].dropna().str.strip()
    as_number = pd.to_numeric(values.str.replace(",", "", regex=False), errors="coerce")
    as_date = pd.to_datetime(values, format="%d.%m.%Y", errors="coerce")
    scan.append({
        "Kolona": col,
        "Vlera": len(values),
        "% numër": round(as_number.notna().mean() * 100, 2),
        "% datë": round(as_date.notna().mean() * 100, 2),
    })
scan = pd.DataFrame(scan)
scan["Tipi i vërtetë"] = scan.apply(
    lambda r: "numerik" if r["% numër"] > 95 else ("datë" if r["% datë"] > 95 else "tekst / kategorik"), axis=1)
scan

,Kolona,Vlera,% numër,% datë,Tipi i vërtetë
0,Autoriteti_kontraktues,277764,0.00,0.0,tekst / kategorik
1,Numri_i_references,277764,0.00,0.0,tekst / kategorik
2,Objekti_i_prokurimit,277761,0.01,0.0,tekst / kategorik
3,Lloji_i_procedures,277764,0.00,0.0,tekst / kategorik
4,Tipi_i_kontrates,277764,0.00,0.0,tekst / kategorik
5,Lloji_i_marreveshjes_kuader,19424,0.00,0.0,tekst / kategorik
6,Fondi_limit,252200,100.00,0.0,numerik
7,Data_e_publikimit,277764,0.00,100.0,datë
8,Data_e_hapjes,277595,0.00,100.0,datë
9,Data_e_mbylljes,276672,0.00,100.0,datë


## 2. Definimi i tipeve të të dhënave

Çdo pjesë ka përcaktuar tipin e kolonave të veta: kategorik (nominal, ordinal, binar, tekst) ose numerik (interval, ratio), diskret ose i vazhdueshëm. Kolona `csv_type` tregon që **të gjitha** kolonat ruhen si tekst në CSV, pra tipi real duhet vendosur në pastrim.

**Shkallët e matjes:** nominale (p.sh. autoriteti, fituesi), ordinale (p.sh. kategoria e fondit), **intervale** (datat: diferenca mes dy datave ka kuptim, por nuk ka zero absolute) dhe raporti / ratio (shumat në lekë, numri i ofertave: kanë zero absolute).

In [46]:
from src.common.data_types import describe_types
from src.procedure_competition.columns import COLUMNS as PC_COLUMNS
from src.time_integration.columns import COLUMNS as TI_COLUMNS
from src.values_winners.columns import COLUMNS as VW_COLUMNS

PARTS = {
    "Vlerat dhe fituesit": VW_COLUMNS,
    "Koha dhe integrimi": {c: t for c, t in TI_COLUMNS.items() if c in raw},
    "Procedura dhe konkurrenca": PC_COLUMNS,
}
types = pd.concat([describe_types(raw, cols).assign(pjesa=name) for name, cols in PARTS.items()],
                  ignore_index=True)
types[["pjesa", "column", "category", "type", "nature", "csv_type", "unique_values"]]

,pjesa,column,category,type,nature,csv_type,unique_values
0,Vlerat dhe fituesit,Fondi_limit,Numeric,Ratio,Continuous,text,102064
1,Vlerat dhe fituesit,Vlera_e_fituesit,Numeric,Ratio,Continuous,text,70706
2,Vlerat dhe fituesit,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Numeric,Ratio,Continuous,text,37018
3,Vlerat dhe fituesit,Lidhja_e_kontrates_me_TVSH,Categorical,Binary (Po/Jo),Discrete,text,2
4,Vlerat dhe fituesit,Fituesi,Categorical,Nominal,Discrete,text,8269
5,Vlerat dhe fituesit,NIPT_i_fituesit,Categorical,Nominal (identifier),Discrete,text,8305
6,Koha dhe integrimi,Data_e_publikimit,Numeric,Interval (date),Discrete,text,3320
7,Koha dhe integrimi,Data_e_hapjes,Numeric,Interval (date),Discrete,text,3353
8,Koha dhe integrimi,Data_e_mbylljes,Numeric,Interval (date),Discrete,text,3193
9,Koha dhe integrimi,Numri_i_references,Categorical,Nominal (identifier),Discrete,text,277757


## 3. Kualiteti i të dhënave

Në Fazën I problemet **vetëm identifikohen dhe dokumentohen**; asnjë rresht nuk ndryshohet këtu. Çdo pjesë kontrollon kolonat e veta sipas dimensioneve të leksionit.

In [47]:
# Vlerat që mungojnë në të dhënat bruto, me përqindje
missing = pd.DataFrame({
    "Vlerat_munguese": raw.isna().sum(),
    "Perqindja_%": (raw.isna().mean() * 100).round(2),
})
missing[missing["Vlerat_munguese"] > 0].sort_values("Perqindja_%", ascending=False)

,Vlerat_munguese,Perqindja_%
Lloji_i_marreveshjes_kuader,258340,93.01
Vlera_e_fituesit_ne_lidhjen_e_kontrates,226259,81.46
Arsyeja_e_anullimit,205859,74.11
Kodet_CPV,61688,22.21
Vlera_e_fituesit,57382,20.66
Fituesi,53894,19.40
NIPT_i_fituesit,53880,19.40
Fondi_limit,25564,9.20
Data_e_mbylljes,1092,0.39
Data_e_hapjes,169,0.06


In [48]:
# Duplikatet: rreshta krejtësisht të njëjtë, dhe numra reference të përsëritur
print("Rreshta të dyfishuar (të 21 kolonat):", raw.drop(columns="year").duplicated().sum())
ref_dupes = raw[raw["Numri_i_references"].duplicated(keep=False)]
print("Procedura me referencë të përsëritur:", len(ref_dupes), "| referenca:", ref_dupes["Numri_i_references"].nunique())
ref_dupes.sort_values("Numri_i_references")[["Numri_i_references", "year", "Autoriteti_kontraktues", "Fituesi"]].head(6)

Rreshta të dyfishuar (të 21 kolonat): 0
Procedura me referencë të përsëritur: 14 | referenca: 7


,Numri_i_references,year,Autoriteti_kontraktues,Fituesi
240345,REF-20571-09-19-2024,2024,Agjencia Kombetare e Planifikimit te Territorit,INFOSOFT OFFICE
241189,REF-20571-09-19-2024,2024,Drejtoria e Sherbimit Spitalor Skrapar,NOART
6042,REF-22877-02-04-2016,2016,Spitali Obsetrik Gjinekologjik Mbreteresha Geraldine,SUPPORT - 07 SH.P.K
9552,REF-22877-02-04-2016,2016,Klubi Sportiv Albpetrol Patos,ERJON BEGAJ
4290,REF-35545-04-27-2016,2016,Instituti i Mjekesise Ligjore,MOBILERI CURRI
17228,REF-35545-04-27-2016,2016,Hekurudha Shqiptare sh.a,TECH INVEST


Nuk ka rreshta të dyfishuar, prandaj `drop_duplicates()` nuk heq asgjë. Referencat e përsëritura i përkasin autoriteteve të ndryshme, pra janë procedura të ndryshme dhe **nuk fshihen**; një procedurë identifikohet nga referenca **bashkë me** autoritetin.

Më poshtë, kontrollet e plota të çdo pjese, sipas dimensioneve të kualitetit.

In [49]:
from src.common.quality import run_checks
from src.procedure_competition.prepare import quality_report
from src.time_integration.quality import build_checks as time_checks
from src.values_winners.quality import build_checks as value_checks

quality = pd.concat([
    run_checks(raw, value_checks(raw)).assign(pjesa="Vlerat dhe fituesit"),
    time_checks(raw).assign(pjesa="Koha dhe integrimi"),  # e kthen tabelën të gatshme
    quality_report(raw[list(PC_COLUMNS)]).assign(pjesa="Procedura dhe konkurrenca"),
], ignore_index=True)

print(f"{len(quality)} kontrolle, {int((quality['rows'] > 0).sum())} me probleme")
quality.groupby(["pjesa", "dimension"]).size().unstack(fill_value=0)

80 kontrolle, 46 me probleme


dimension,Accuracy,Completeness,Consistency,Format,Uniqueness
pjesa,,,,,
Koha dhe integrimi,7,4,4,6,0
Procedura dhe konkurrenca,2,12,2,13,1
Vlerat dhe fituesit,8,6,4,8,3


In [50]:
# Problemet kryesore: kontrollet me më shumë rreshta të prekur
quality[quality["rows"] > 0].sort_values("rows", ascending=False).head(15)[
    ["pjesa", "dimension", "column", "check", "rows", "pct_of_base"]]

,pjesa,dimension,column,check,rows,pct_of_base
54,Procedura dhe konkurrenca,Completeness,Lloji_i_marreveshjes_kuader,Missing or blank value,258340,93.01
5,Vlerat dhe fituesit,Completeness,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Missing value,226259,81.46
68,Procedura dhe konkurrenca,Format,Kodet_CPV,Leading/trailing or repeated whitespace,216076,100.00
56,Procedura dhe konkurrenca,Completeness,Arsyeja_e_anullimit,Missing or blank value,205859,74.11
79,Procedura dhe konkurrenca,Format,Kodet_CPV,More than one CPV code in cell,61968,28.68
60,Procedura dhe konkurrenca,Completeness,Kodet_CPV,Missing or blank value,61688,22.21
4,Vlerat dhe fituesit,Completeness,Vlera_e_fituesit,Missing value,57382,20.66
7,Vlerat dhe fituesit,Completeness,Fituesi,Missing value,53894,19.40
8,Vlerat dhe fituesit,Completeness,NIPT_i_fituesit,Missing value,53880,19.40
63,Procedura dhe konkurrenca,Format,Objekti_i_prokurimit,Leading/trailing or repeated whitespace,37462,13.49


**Dimensionet e tjera të leksionit:**

- **Believability (besueshmëria):** burimi është institucion shtetëror (APP), i detyruar me ligj të publikojë procedurat; nuk ka ndërhyrje manuale në skedarë.
- **Timeliness (aktualiteti):** eksporti është i tetorit 2026 dhe mbulon 2016–2025; viti 2026 përjashtohet sepse ishte i paplotë. Disa fusha plotësohen vetëm nga një vit i caktuar (p.sh. TVSH nga 2018), gjë që duhet mbajtur parasysh kur krahasohen vitet.

---
# b) Pastrimi, vlerat e zbrazëta, integrimi, agregimi dhe mostrimi

## 4. Pastrimi

Pastrimi rregullon formatin dhe konsistencën **pa fshirë asnjë rresht**. Vlerat logjikisht të pamundura (p.sh. vlera e fituesit mbi fondin) nuk ndryshohen: ato i takojnë Pjesës II.

In [51]:
from src.values_winners.cleaning import clean as clean_values, cleaning_summary as value_cleaning

values = clean_values(raw)
value_cleaning(raw, values)

,column,step,before,after
0,Fondi_limit,Text amount -> number,"text (252,200 values)","float64 (252,200 values)"
1,Vlera_e_fituesit,Text amount -> number,"text (220,382 values)","float64 (220,382 values)"
2,Vlera_e_fituesit_ne_lidhjen_e_kontrates,Text amount -> number,"text (51,505 values)","float64 (51,505 values)"
3,Lidhja_e_kontrates_me_TVSH,Po/Jo -> boolean,"{'Jo': 251197, 'Po': 26567}","{'False': np.int64(251197), 'True': np.int64(26567)}"
4,NIPT_i_fituesit,"Trim whitespace/newlines, upper case","8,305 distinct","7,444 distinct"
5,Fituesi,"Trim whitespace/newlines, upper case, one name per NIPT","8,269 distinct","7,403 distinct"


In [52]:
from src.time_integration.cleaning import clean as clean_dates, cleaning_summary as date_cleaning, date_order_checks
from src.time_integration.columns import INPUT_COLS
from src.time_integration.missing_values import fill_missing_dates, parse_dates

# Lloji i procedurës lexohet vetëm për plotësimin e datave (mediana sipas llojit x vitit)
dates_raw = parse_dates(raw[INPUT_COLS + ["year"]])
dates = clean_dates(dates_raw)
display(date_cleaning(dates))

# Renditja e datave para dhe pas pastrimit
pd.DataFrame({"para": date_order_checks(dates_raw), "pas": date_order_checks(fill_missing_dates(dates))})

,column,problem,action,rows
0,Data_e_publikimit,"Opening before publication, after registration",Publication date moved back to the opening date,7110
1,Data_e_hapjes,Opening before registration or after closing,"Set to missing, then filled (publication + median delay)",65
2,Data_e_mbylljes,Closing before registration or after the export date (19.10.2026),"Set to missing, then filled (opening + median duration)",6


,para,pas
Publication before registration,0,0
Opening before publication,7170,0
Closing before opening,5,0
Closing before publication,245,0
Closing after export date,1,0
Tender duration > 365 days,5,4


In [53]:
from src.procedure_competition.prepare import clean_and_engineer

procedures = clean_and_engineer(raw[[*PC_COLUMNS, "year"]])
after = quality_report(procedures, cleaned=True)
after[after["dimension"] == "Format"][["column", "check", "rows"]]

,column,check,rows
12,Autoriteti_kontraktues,Leading/trailing or repeated whitespace,0
13,Objekti_i_prokurimit,Leading/trailing or repeated whitespace,0
14,Lloji_i_procedures,Leading/trailing or repeated whitespace,0
15,Tipi_i_kontrates,Leading/trailing or repeated whitespace,0
16,Lloji_i_marreveshjes_kuader,Leading/trailing or repeated whitespace,0
17,Arsyeja_e_anullimit,Leading/trailing or repeated whitespace,0
18,Kodet_CPV,Leading/trailing or repeated whitespace,0
19,Anulluar,Value outside boolean vocabulary,0
20,Pezulluar,Value outside boolean vocabulary,0
21,Numri_i_ofertave_te_dorezuara,Non-numeric value,0


## 5. Vlerat e zbrazëta: identifikimi dhe strategjia

Si në leksion, **të dhënat që mungojnë nuk fshihen**. Fondet 0 ose ≤ 1 lek janë **mungesa të maskuara** (vlera të paracaktuara kur fondi nuk publikohet) dhe shënohen si mungesë.

In [54]:
from src.common.missing import compare_numeric_strategies, missing_summary
from src.values_winners.missing_values import mark_disguised_missing

values_m = mark_disguised_missing(values)
missing_summary(values, VW_COLUMNS).merge(
    missing_summary(values_m, VW_COLUMNS), on="column", suffixes=("_bruto", "_me_te_maskuarat"))

,column,missing_bruto,missing_pct_bruto,missing_me_te_maskuarat,missing_pct_me_te_maskuarat
0,Fondi_limit,25564,9.20,42287,15.22
1,Vlera_e_fituesit,57382,20.66,57382,20.66
2,Vlera_e_fituesit_ne_lidhjen_e_kontrates,226259,81.46,226259,81.46
3,Lidhja_e_kontrates_me_TVSH,0,0.00,0,0.00
4,Fituesi,53894,19.40,53894,19.40
5,NIPT_i_fituesit,53880,19.40,53880,19.40


In [55]:
# Krahasimi i strategjive për fondin: fshirja humb rreshta, imputimi shpik vlera
compare_numeric_strategies(values_m["Fondi_limit"], raw["Lloji_i_procedures"])

,strategy,rows_kept,still_missing,median,mean
0,Original (with NaN),277764,42287,390000.0,9150130.36
1,Deletion (drop rows),235477,0,390000.0,9150130.36
2,Median imputation,277764,0,390000.0,7816481.53
3,Median by group,277764,0,263277.0,7789468.91
4,Keep NaN + missing flag,277764,42287,390000.0,9150130.36


**Strategjitë e zgjedhura:**
- **Vlerat dhe fituesit:** `NaN` + flag, pa imputim. 85% e procedurave pa fitues janë të anuluara dhe 83.5% e blerjeve të biletave nuk kanë fond, pra mungesat kanë shpjegim (MAR) dhe një vlerë e shpikur do të prishte analizën.
- **Koha:** datat e hapjes/mbylljes që mungojnë plotësohen me datën paraprake + medianën e vonesës së procedurave të të njëjtit lloj në të njëjtin vit (strategjia më e saktë nga 4 të krahasuara), me flag `*_imputed` (mungesat janë pak: 0.06% dhe 0.39%).
- **Procedura dhe konkurrenca:** mungesat ruhen si mungesa; nuk imputohen kategori të hamendësuara.

In [56]:
from src.time_integration.missing_values import TIME_COLS, imputation_report

imputation_report(fill_missing_dates(dates), missing_summary(dates, TIME_COLS))

,column,missing_before,missing_pct_before,missing_after,missing_pct_after,imputed
0,Data_e_publikimit,0,0.0,0,0.0,0
1,Data_e_hapjes,0,0.0,0,0.0,0
2,Data_e_mbylljes,0,0.0,0,0.0,0
3,Numri_i_references,0,0.0,0,0.0,0


In [57]:
from src.procedure_competition.prepare import missing_value_report

missing_value_report(raw[list(PC_COLUMNS)], procedures)

,column,missing_before,pct_before,missing_after,pct_after,treatment
0,Autoriteti_kontraktues,0,0.00,0,0.00,Whitespace normalized; blank values retained as missing
1,Objekti_i_prokurimit,3,0.00,3,0.00,Whitespace normalized; blank values retained as missing
2,Lloji_i_procedures,0,0.00,0,0.00,Whitespace normalized; blank values retained as missing
3,Tipi_i_kontrates,0,0.00,0,0.00,Whitespace normalized; blank values retained as missing
4,Lloji_i_marreveshjes_kuader,258340,93.01,258340,93.01,Retained as missing; no reliable source value to impute
5,Anulluar,0,0.00,0,0.00,Mapped Po/Jo to boolean; invalid or blank values retained as missing
6,Arsyeja_e_anullimit,205859,74.11,205859,74.11,Retained as missing; only applicable to cancelled procedures
7,Pezulluar,0,0.00,0,0.00,Mapped Po/Jo to boolean; invalid or blank values retained as missing
8,Numri_i_ofertave_te_dorezuara,0,0.00,0,0.00,Converted to nullable integer; missing values retained
9,Numri_i_ofertave_te_kualifikuara,0,0.00,0,0.00,Converted to nullable integer; missing values retained


## 6. Integrimi

Tri pjesët përgatiten nga i njëjti lexim dhe bashkohen **një-me-një** mbi çelësin `Numri_i_references` + `Autoriteti_kontraktues` (referenca vetëm nuk është unike: 7 referenca përdoren nga dy procedura). Integrohen edhe dy burime të jashtme: **datat e zgjedhjeve (KQZ)** dhe **ndarjet CPV** (Rregullorja EC 213/2008).

In [58]:
from src.time_integration.integrated_dataset import column_sources, integrate, integration_summary, prepare_parts

parts = prepare_parts(raw)
integrated = integrate(parts)
integration_summary(raw, integrated, column_sources(integrated, parts))

,metric,value
0,Raw rows (10 yearly files),277764
1,Integrated rows,277764
2,Rows lost or duplicated,0
3,Duplicated procedure keys,0
4,Columns,64
5,Columns from key,3
6,Columns from values_winners,15
7,Columns from time_integration,23
8,Columns from procedure_competition,23
9,Years,2016-2025


## 7. Agregimi

Shumat përmblidhen me **medianë**: shuma dhe mesatarja do të dominoheshin nga vlerat e pamundura (deri në 35 mijë miliardë lekë) që trajtohen në Pjesën II. Interpretimi i dallimeve bëhet në Pjesën II.

In [59]:
from src.values_winners.aggregation import by_group, concentration, winners

by_group(integrated, "year")

,year,procedures,with_winner_pct,with_winner,consortia,with_vat,with_vat_pct,median_fund,median_winner_value,median_savings_pct
0,2016,33375,80.21,26770,0,6,0.02,250000.00,169875.0,12.400
1,2017,31769,82.31,26150,2,710,2.23,272000.00,176700.0,12.265
2,2018,33880,73.23,24812,0,4593,13.56,297390.00,199700.0,8.290
3,2019,28313,76.83,21753,0,3603,12.73,356400.00,199000.0,9.090
4,2020,23670,81.39,19266,10,3349,14.15,400000.00,244880.0,8.530
5,2021,22653,84.05,19039,28,2734,12.07,404050.00,270050.0,8.100
6,2022,26034,80.94,21071,38,2885,11.08,468656.00,293000.0,8.000
7,2023,25137,83.57,21006,22,2635,10.48,499000.00,280000.0,10.240
8,2024,27466,81.96,22511,25,3274,11.92,560606.67,300000.0,11.130
9,2025,25467,84.39,21492,44,2778,10.91,500000.00,289850.0,13.580


In [60]:
by_group(integrated, "afersia_me_zgjedhjet")

,afersia_me_zgjedhjet,procedures,with_winner_pct,with_winner,consortia,with_vat,with_vat_pct,median_fund,median_winner_value,median_savings_pct
0,Mbi 4 muaj para,95986,80.01,76802,61,9681,10.09,375000.0,230000.0,9.90
1,Periudha parazgjedhore (4 muaj),47804,80.20,38340,36,4585,9.59,441859.0,257890.0,10.06
2,Deri 4 muaj pas,40002,82.77,33109,33,3812,9.53,372000.0,230000.0,10.68
3,Mbi 4 muaj pas,93972,80.47,75619,39,8489,9.03,372050.0,226000.0,10.41


In [61]:
# Përqendrimi i tregut sipas numrit të kontratave
concentration(winners(integrated))

,winners,count,contracts,share_pct
0,All,7444,223884,100.00
1,Top 1,1,5762,2.57
2,Top 10,10,22368,9.99
3,Top 100,100,82146,36.69


### Agregimi kohor me `resample`

Data e publikimit vendoset si indeks, që Pandas të mund të grupojë sipas periudhave kohore. Krahasohet niveli i agregimit: sa më e gjatë periudha, aq më pak vlera dhe aq më e qetë seria. Viti këtu është viti i **publikimit**, jo i skedarit: disa procedura publikohen pas vitit të skedarit (p.sh. regjistruar në dhjetor, publikuar në janar).

In [62]:
by_date = integrated.set_index("Data_e_publikimit").sort_index()

# 223 procedura kanë datë publikimi pas vitit të skedarit (deri në shtator 2026);
# agregimi kohor kufizohet te periudha e projektit 2016-2025
outside = by_date.loc["2026":]
print(f"Procedura të publikuara pas 2025 (lihen jashtë agregimit kohor): {len(outside)}")
by_date = by_date.loc["2016":"2025"]

monthly = by_date.resample("MS").agg(procedura=("Fondi_limit", "size"), mediana_e_fondit=("Fondi_limit", "median"))
quarterly = by_date.resample("QS").agg(procedura=("Fondi_limit", "size"), mediana_e_fondit=("Fondi_limit", "median"))
yearly = by_date.resample("YS").agg(procedura=("Fondi_limit", "size"), mediana_e_fondit=("Fondi_limit", "median"))

display(monthly.head(12))
pd.DataFrame({
    "Numri i vlerave": [len(by_date), len(monthly), len(quarterly), len(yearly)],
    "Procedura mesatarisht në periudhë": [1, monthly["procedura"].mean(), quarterly["procedura"].mean(),
                                          yearly["procedura"].mean()],
}, index=["Procedurat", "Agregimi mujor", "Agregimi tremujor", "Agregimi vjetor"]).round(1)

Procedura të publikuara pas 2025 (lihen jashtë agregimit kohor): 15


,procedura,mediana_e_fondit
Data_e_publikimit,,
2016-01-01,1045,352480.00
2016-02-01,2965,301394.00
2016-03-01,3407,303026.00
2016-04-01,3384,300000.00
2016-05-01,3178,290163.69
2016-06-01,2832,250000.00
2016-07-01,2149,300000.00
2016-08-01,2053,285471.00
2016-09-01,2641,213354.00


,Numri i vlerave,Procedura mesatarisht në periudhë
Procedurat,277749,1.0
Agregimi mujor,120,2314.6
Agregimi tremujor,40,6943.7
Agregimi vjetor,10,27774.9


## 8. Mostrimi

Mostra merr 10% të procedurave. U krahasuan tri metoda; zgjidhet mostra **e shtresëzuar sipas vitit dhe afërsisë me zgjedhjet**, që ruan saktësisht përqindjen e çdo viti dhe të periudhave zgjedhore. Këtu mostrat nxirren nga dataset-i i integruar.

In [63]:
from src.common.sampling import compare_distributions
from src.time_integration.sampling import draw_samples

samples = draw_samples(integrated)
display(pd.DataFrame({name: [len(s), round(len(s) / len(integrated) * 100, 2)]
                      for name, s in samples.items()}, index=["rreshta", "% e dataset-it"]))

comparison = compare_distributions(integrated, samples, ["year", "afersia_me_zgjedhjet"])
comparison[comparison["variable"] == "afersia_me_zgjedhjet"]

,random,stratified_by_year,stratified_by_year_election
rreshta,27776.0,27777.0,27774.0
% e dataset-it,10.0,10.0,10.0


,variable,value,population,random,stratified_by_year,stratified_by_year_election
10,afersia_me_zgjedhjet,Mbi 4 muaj para,34.56,34.28,34.55,34.56
11,afersia_me_zgjedhjet,Periudha parazgjedhore (4 muaj),17.21,17.73,16.99,17.21
12,afersia_me_zgjedhjet,Deri 4 muaj pas,14.40,14.47,14.52,14.40
13,afersia_me_zgjedhjet,Mbi 4 muaj pas,33.83,33.52,33.95,33.83


In [64]:
# Devijimi më i madh nga dataset-i i plotë, mesatarisht në 50 seed-e (pikë përqindjeje)
pd.read_csv("reports/time_integration/sampling_max_deviation.csv")

,Unnamed: 0,mean,mean.1,mean.2,worst,worst.1,worst.2
0,NaN,random,stratified_by_year,stratified_by_year_election,random,stratified_by_year,stratified_by_year_election
1,variable,NaN,NaN,NaN,NaN,NaN,NaN
2,year,0.32,0.01,0.01,0.61,0.01,0.01
3,periudha_covid,0.32,0.06,0.05,0.74,0.18,0.12
4,afersia_me_zgjedhjet,0.31,0.32,0.0,0.68,0.71,0.0
5,kategoria_e_kohezgjatjes,0.27,0.3,0.31,0.76,0.76,0.67
6,periudhe_parazgjedhore,0.15,0.17,0.0,0.52,0.47,0.0
7,publikim_ne_dhjetor,0.14,0.15,0.12,0.35,0.58,0.48
8,hapje_mbyllje_e_njejta_dite,0.1,0.09,0.12,0.3,0.28,0.36


### Reservoir sampling

Kjo metodë zgjedh një mostër me madhësi fikse **k** nga një rrjedhë të dhënash, duke e lexuar çdo rresht vetëm një herë (p.sh. kur të dhënat nuk nxënë në memorie). Rreshti i *i*-të zëvendëson një element të mostrës me probabilitet *k / i*, kështu që në fund çdo rresht ka të njëjtën gjasë të jetë në mostër.

In [65]:
import numpy as np

def reservoir_sample(n_rows, k, seed=42):
    rng = np.random.default_rng(seed)
    reservoir = list(range(k))           # k rreshtat e parë hyjnë direkt
    for i in range(k, n_rows):           # çdo rresht tjetër lexohet një herë
        j = rng.integers(0, i + 1)
        if j < k:                        # zëvendëson me probabilitet k / (i + 1)
            reservoir[j] = i
    return reservoir

k = round(len(integrated) * 0.10)
reservoir = integrated.iloc[reservoir_sample(len(integrated), k)]

# Krahasimi i përqindjes sipas vitit me dataset-in e plotë dhe me mostrat e tjera
pd.DataFrame({
    "Dataset-i i plotë": integrated["year"].value_counts(normalize=True),
    "Reservoir": reservoir["year"].value_counts(normalize=True),
    **{name: s["year"].value_counts(normalize=True) for name, s in samples.items()},
}).mul(100).round(2).sort_index()

,Dataset-i i plotë,Reservoir,random,stratified_by_year,stratified_by_year_election
year,,,,,
2016,12.02,12.07,11.86,12.02,12.01
2017,11.44,11.53,11.75,11.44,11.44
2018,12.20,12.11,12.35,12.20,12.20
2019,10.19,10.11,10.20,10.19,10.19
2020,8.52,8.34,8.51,8.52,8.52
2021,8.16,8.37,8.33,8.15,8.16
2022,9.37,9.39,9.25,9.37,9.37
2023,9.05,9.07,9.07,9.05,9.05
2024,9.89,10.10,9.77,9.89,9.89


Reservoir sampling sillet si mostra e rastësishme e thjeshtë: përqindjet luhaten pak rreth dataset-it të plotë. Mostra e shtresëzuar sipas vitit dhe zgjedhjeve i ruan saktësisht, prandaj ajo mbetet zgjedhja e projektit.

---
# c) Krijimi i vetive, diskretizimi, binarizimi, transformimi, zgjedhja e vetive dhe reduktimi

## 9. Krijimi i vetive

In [66]:
from src.time_integration.features import features_summary as time_features
from src.values_winners.features import features_summary as value_features

display(value_features(integrated)[["feature", "step", "definition", "non_null"]])
time_features(integrated)[["feature", "group", "definition", "non_null"]]

,feature,step,definition,non_null
0,kursimi,Created,Fondi_limit - Vlera_e_fituesit (lekë),201837
1,kursimi_pct,Created,kursimi / Fondi_limit * 100,201837
2,nr_fituesve,Created,Number of winners in the cell (consortium members),223870
3,fitues_i_perbashket,Created + binarized,1 if the contract was won by a consortium,223870
4,kategoria_e_fondit,Discretized,Fondi_limit in 4 equal-frequency bins,235477
5,me_tvsh,Binarized,Lidhja_e_kontrates_me_TVSH: Po/Jo -> 1/0,277764
6,Fondi_limit_missing,Binarized,Missing-value flag -> 1/0,277764
7,Vlera_e_fituesit_missing,Binarized,Missing-value flag -> 1/0,277764
8,Fituesi_missing,Binarized,Missing-value flag -> 1/0,277764


,feature,group,definition,non_null
0,muaji_publikimit,Kohore,Muaji i Data_e_publikimit (1-12),277764
1,tremujori_publikimit,Kohore,Tremujori i Data_e_publikimit (1-4),277764
2,dita_e_javes_publikimit,Kohore,Dita e javës e Data_e_publikimit (0 = e hënë),277764
3,ditet_publikim_hapje,Kohore,Data_e_hapjes - Data_e_publikimit (ditë),277764
4,kohezgjatja_e_tenderit,Kohore,Data_e_mbylljes - Data_e_hapjes (ditë),277764
5,zgjedhjet_me_te_aferta,Zgjedhore,Data e zgjedhjeve më afër Data_e_publikimit,277764
6,ditet_nga_zgjedhjet,Zgjedhore,"Data_e_publikimit - zgjedhjet_me_te_aferta (ditë, < 0 = para)",277764
7,periudha_covid,COVID,"Para (< 11.03.2020), Gjatë (deri 31.12.2021), Pas COVID",277764
8,kategoria_e_kohezgjatjes,Diskretizuar,kohezgjatja_e_tenderit në 5 grupe me kufij të zgjedhur,277764
9,afersia_me_zgjedhjet,Diskretizuar,Data_e_publikimit në 4 grupe (±4 muaj nga zgjedhjet),277764


In [67]:
# Vetitë e konkurrencës (procedura dhe konkurrenca)
competition = ["cpv_count", "primary_cpv_division_name", "qualified_offer_share", "single_bidder",
               "no_qualified_offers", "competition_band", "is_framework_agreement",
               "has_cancellation_reason", "cancelled_without_reason"]
pd.DataFrame({"non_null": integrated[competition].notna().sum(), "dtype": integrated[competition].dtypes.astype(str)})

,non_null,dtype
cpv_count,277764,Int64
primary_cpv_division_name,216076,string
qualified_offer_share,252020,Float64
single_bidder,277764,boolean
no_qualified_offers,277764,boolean
competition_band,277764,category
is_framework_agreement,277764,boolean
has_cancellation_reason,277764,boolean
cancelled_without_reason,277764,boolean


## 10. Diskretizimi

Leksioni krahason ndarjen me **gjerësi të barabartë** dhe me **frekuencë të barabartë**. Te shumat dhe kohëzgjatjet, gjerësia e barabartë fut pothuajse të gjitha procedurat në intervalin e parë, sepse disa vlera shumë të mëdha e zgjerojnë intervalin.

In [68]:
from src.common.features import compare_discretizations

# Fondi limit: zgjidhet frekuenca e barabartë (kuartilet)
compare_discretizations(integrated["Fondi_limit"], 4)

,bin,equal_width_interval,equal_width_rows,equal_frequency_interval,equal_frequency_rows
0,1,"(-39319998.949, 9830000000.787]",235466,"(1.0490000000000002, 134964.0]",58870
1,2,"(9830000000.787, 19660000000.525]",5,"(134964.0, 390000.0]",58980
2,3,"(19660000000.525, 29490000000.262]",1,"(390000.0, 1000000.0]",61425
3,4,"(29490000000.262, 39320000000.0]",5,"(1000000.0, 39320000000.0]",56202


In [69]:
from src.time_integration.features import DURATION_BINS, DURATION_LABELS, discretization_comparison

# Kohëzgjatja: 53% e tenderave zgjasin 1 ditë, ndaj kuantilet përsëriten -> kufij të zgjedhur
discretization_comparison(integrated["kohezgjatja_e_tenderit"], 5, DURATION_BINS, DURATION_LABELS)

,method,bin,rows
0,Equal width,"(-0.44, 88.0]",277167
1,Equal width,"(88.0, 176.0]",549
2,Equal width,"(176.0, 264.0]",32
3,Equal width,"(264.0, 352.0]",12
4,Equal width,"(352.0, 440.0]",4
0,Equal frequency,"(-0.001, 1.0]",161150
1,Equal frequency,"(1.0, 2.0]",13420
2,Equal frequency,"(2.0, 11.0]",55378
3,Equal frequency,"(11.0, 440.0]",47816
0,Chosen limits,E njëjta ditë,16076


In [70]:
# Numri i ofertave në 4 grupe
integrated["competition_band"].value_counts(dropna=False).rename("procedura").to_frame()

,procedura
competition_band,
4+ offers,168411
2-3 offers,46334
Single offer,37275
No offers,25744


## 11. Binarizimi

In [71]:
from src.time_integration.features import binary_summary

value_binaries = ["me_tvsh", "fitues_i_perbashket", "Fondi_limit_missing",
                  "Vlera_e_fituesit_missing", "Fituesi_missing"]
pd.concat([
    pd.DataFrame([{"feature": f, "ones": int(integrated[f].sum()),
                   "pct_ones": round(integrated[f].mean() * 100, 2)} for f in value_binaries]),
    binary_summary(integrated),
], ignore_index=True)

,feature,ones,pct_ones
0,me_tvsh,26567,9.56
1,fitues_i_perbashket,169,0.08
2,Fondi_limit_missing,42287,15.22
3,Vlera_e_fituesit_missing,57382,20.66
4,Fituesi_missing,53894,19.40
5,gjate_covid,42198,15.19
6,periudhe_parazgjedhore,47804,17.21
7,publikim_ne_fundjave,2781,1.00
8,publikim_ne_dhjetor,25623,9.22
9,hapje_mbyllje_e_njejta_dite,16076,5.79


## 12. Transformimi

Shumat në lekë kanë asimetri shumë të madhe. Fillimisht aplikohet **log(1 + x)**, pastaj **min-max** (0–1) dhe **z-score** (mesatarja 0, devijimi 1). Min-max pa log nuk funksionon: pothuajse të gjitha vlerat grumbullohen afër 0. Kategoritë e procedurave kodohen me **one-hot** dhe **frequency encoding**.

In [72]:
from src.values_winners.transformation import transform

_, transformation = transform(integrated)
transformation[["column", "transformation", "skew_before", "min_after", "max_after",
                "mean_after", "std_after", "skew_after"]]

,column,transformation,skew_before,min_after,max_after,mean_after,std_after,skew_after
0,Fondi_limit,log(1 + x),135.74,0.7178,24.3950,12.9589,2.1704,0.02
1,Fondi_limit,min-max (without log),135.74,0.0000,1.0000,0.0002,0.0052,135.74
2,Fondi_limit,log + min-max,135.74,0.0000,1.0000,0.5170,0.0917,0.02
3,Fondi_limit,log + z-score,135.74,-5.6400,5.2691,0.0000,1.0000,0.02
4,Vlera_e_fituesit,log(1 + x),469.45,0.0000,31.1864,12.3591,2.2128,-0.87
5,Vlera_e_fituesit,min-max (without log),469.45,0.0000,1.0000,0.0000,0.0021,469.45
6,Vlera_e_fituesit,log + min-max,469.45,0.0000,1.0000,0.3963,0.0710,-0.87
7,Vlera_e_fituesit,log + z-score,469.45,-5.5852,8.5082,0.0000,1.0000,-0.87
8,Vlera_e_fituesit_ne_lidhjen_e_kontrates,log(1 + x),226.95,0.0000,31.1864,13.9816,3.0382,-1.30
9,Vlera_e_fituesit_ne_lidhjen_e_kontrates,min-max (without log),226.95,0.0000,1.0000,0.0000,0.0044,226.95


In [73]:
from src.procedure_competition.encoding import encode_categories

encoded, encoding = encode_categories(procedures)
encoding

,source_column,encoding_method,input_categories,generated_columns,missing_policy
0,Lloji_i_procedures,one_hot,13,13,Separate [MISSING] category when present
1,Tipi_i_kontrates,one_hot,3,3,Separate [MISSING] category when present
2,Lloji_i_marreveshjes_kuader,one_hot,4,4,Separate [MISSING] category when present
3,primary_cpv_division,one_hot,46,46,Separate [MISSING] category when present
4,competition_band,one_hot,4,4,Separate [MISSING] category when present
5,Autoriteti_kontraktues,frequency,1786,1,Missing values treated as a separate category
6,primary_cpv_code,frequency,4454,1,Missing values treated as a separate category
7,Objekti_i_prokurimit,not_encoded_free_text,189448,0,Original text and missing values preserved
8,Arsyeja_e_anullimit,not_encoded_free_text,22853,0,Original text preserved; indicator feature used


## 13. Zgjedhja e nën-bashkësisë së vetive dhe korrelacioni

Për procedurat dhe konkurrencën, kolonat tekstuale dhe kategoritë e zëvendësuara nga encoding-u përjashtohen. Për PCA-në zgjidhen vetitë sasiore të të tri pjesëve; **korrelacioni** tregon vetitë që mbartin të njëjtin informacion (p.sh. ofertat e dorëzuara dhe të kualifikuara: 0.86).

In [74]:
from src.procedure_competition.feature_selection import select_analysis_features

_, selection = select_analysis_features(encoded)
display(selection["status"].value_counts().to_frame())
selection[selection["status"] == "excluded"][["column", "reason"]]

,count
status,
selected,87
excluded,11


,column,reason
1,Autoriteti_kontraktues,Replaced by contracting_authority_frequency
2,Objekti_i_prokurimit,Free text retained only in the cleaned dataset
3,Lloji_i_procedures,Replaced by procedure_type one-hot columns
4,Tipi_i_kontrates,Replaced by contract_type one-hot columns
5,Lloji_i_marreveshjes_kuader,Replaced by framework_type one-hot columns
7,Arsyeja_e_anullimit,Free text replaced by cancellation indicators
11,Kodet_CPV,Raw multi-value field replaced by parsed CPV features
14,primary_cpv_code,Replaced by primary_cpv_code_frequency
15,primary_cpv_division,Replaced by cpv_division one-hot columns
16,primary_cpv_division_name,Descriptive label retained in the cleaned dataset


In [75]:
from src.values_winners.pca import FEATURES, feature_matrix

X = feature_matrix(integrated)  # log + z-score, vetëm rreshtat me të gjitha vetitë
print(f"{len(X):,} procedura, {X.shape[1]} veti")
X.corr().round(2)

201,837 procedura, 9 veti


,Fondi_limit,Vlera_e_fituesit,ditet_publikim_hapje,kohezgjatja_e_tenderit,ditet_nga_zgjedhjet,Numri_i_ofertave_te_dorezuara,Numri_i_ofertave_te_kualifikuara,qualified_offer_share,cpv_count
Fondi_limit,1.00,0.72,-0.18,0.60,-0.02,-0.11,-0.11,-0.06,0.03
Vlera_e_fituesit,0.72,1.00,-0.15,0.43,-0.03,-0.05,-0.08,-0.11,0.02
ditet_publikim_hapje,-0.18,-0.15,1.00,-0.26,0.05,0.13,0.15,0.06,0.05
kohezgjatja_e_tenderit,0.60,0.43,-0.26,1.00,-0.02,-0.39,-0.39,-0.07,-0.01
ditet_nga_zgjedhjet,-0.02,-0.03,0.05,-0.02,1.00,0.01,0.03,0.05,0.01
Numri_i_ofertave_te_dorezuara,-0.11,-0.05,0.13,-0.39,0.01,1.00,0.86,-0.21,0.14
Numri_i_ofertave_te_kualifikuara,-0.11,-0.08,0.15,-0.39,0.03,0.86,1.00,0.29,0.02
qualified_offer_share,-0.06,-0.11,0.06,-0.07,0.05,-0.21,0.29,1.00,-0.24
cpv_count,0.03,0.02,0.05,-0.01,0.01,0.14,0.02,-0.24,1.00


## 14. Reduktimi i dimensionit (PCA)

In [76]:
from src.common.pca import components_for, pca

explained, loadings, _ = pca(X)
print(f"Komponentë për 80% të variancës: {components_for(explained, 80)}, "
      f"për 90%: {components_for(explained, 90)}")
explained

Komponentë për 80% të variancës: 5, për 90%: 6


,component,eigenvalue,explained_variance_pct,cumulative_pct
0,PC1,2.6270,29.19,29.19
1,PC2,1.6673,18.53,47.71
2,PC3,1.3011,14.46,62.17
3,PC4,1.0239,11.38,73.55
4,PC5,0.8958,9.95,83.50
5,PC6,0.7704,8.56,92.06
6,PC7,0.4644,5.16,97.22
7,PC8,0.2377,2.64,99.86
8,PC9,0.0123,0.14,100.00


In [77]:
loadings[["PC1", "PC2", "PC3", "PC4", "PC5"]]

,PC1,PC2,PC3,PC4,PC5
Fondi_limit,0.4436,-0.4146,-0.1695,-0.0721,0.1341
Vlera_e_fituesit,0.3909,-0.4434,-0.1386,-0.0467,0.1568
ditet_publikim_hapje,-0.2429,0.0349,0.0154,-0.3742,0.8471
kohezgjatja_e_tenderit,0.5043,-0.0816,-0.0620,-0.0541,0.0106
ditet_nga_zgjedhjet,-0.0379,0.0118,-0.1090,-0.8813,-0.4355
Numri_i_ofertave_te_dorezuara,-0.3901,-0.5623,0.0621,0.0875,-0.1337
Numri_i_ofertave_te_kualifikuara,-0.4166,-0.4784,-0.3084,0.0568,-0.0621
qualified_offer_share,-0.0970,0.1721,-0.7188,-0.0524,0.1252
cpv_count,-0.0243,-0.2208,0.5661,-0.2437,0.1127


- **PC1** (29%): madhësia e procedurës kundrejt konkurrencës (fondi, vlera, kohëzgjatja + ; ofertat −).
- **PC4** (11%): koha në raport me zgjedhjet, një dimension pothuajse i pavarur nga të tjerët.

---
# Dataset-i përfundimtar

`prokurimi_final.csv` = dataset-i i integruar + konsistenca mes pjesëve + transformimi + 6 komponentët PCA (≥ 90% e variancës), pa kolonat konstante. Shpjegimi: `docs/final_dataset.md`.

**Konsistenca:** 27,708 procedura janë të anuluara, por kanë fitues. Asgjë nuk ndryshohet; shtohen `kontrate_e_lidhur` (ka fitues dhe nuk është e anuluar) dhe `anulim_kontradiktor` (e anuluar, por me kontratë të lidhur ose me arsyen "asnjë ofertë" ndërkohë që ka oferta). Trajtimi i tyre vendoset në Pjesën II.

In [78]:
from src.final.final_dataset import build

final, tables = build(raw)
display(tables["final_summary"])
tables["final_consistency"]

,metric,value
0,Rows (procedures),277764
1,Rows in the raw files,277764
2,Columns in the integrated dataset,64
3,Columns added,21
4,Constant columns dropped,1
5,Columns in the final dataset,84
6,PCA components kept (>= 90% variance),6
7,Procedures with PCA components,201837


,check,procedures
0,Procedures,277764
1,With a winner,223870
2,Cancelled,73425
3,Cancelled with a winner,27708
4,kontrate_e_lidhur = 1 (winner and not cancelled),196162
5,anulim_kontradiktor = 1,5054
6,of which: cancelled with a signed contract value,3477
7,"of which: reason says no offer was submitted, but offers exist",2001
8,of which: both,424
9,Winner with 0 qualified offers,74


In [79]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
final.to_csv("data/processed/prokurimi_final.csv", index=False)
print(f"U ruajt data/processed/prokurimi_final.csv: {final.shape[0]:,} rreshta x {final.shape[1]} kolona")

U ruajt data/processed/prokurimi_final.csv: 277,764 rreshta x 84 kolona
